In [2]:
import os
os.chdir("../")
os.getcwd()

'/Users/supawitjunsiritrakhoon/Desktop/Customer_Churn_Prediction/Project/customer-churn-prediction'

# Import Libraries

In [9]:
# import necessary libraries
import numpy as np
from pathlib import Path
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from typing import Dict, Any, Union, List
from dateutil.relativedelta import relativedelta

# import project modules
from src.churn_prediction.logger import logger
from src.churn_prediction.pydantic.data_transformation_config import DataTransformationConfig
from src.churn_prediction.pydantic.pipeline_config import PipelineConfig
from src.churn_prediction.utils.common import load_single_config, get_execution_date, get_spark
from src.churn_prediction.utils.loaders import load_data
from src.churn_prediction.utils.writers import write_data

In [3]:
spark = get_spark()

# Feature
1. Station Features (from station_dim - location & location type)
`station_location_feat` or `station_geo_feat`
Or more descriptive: `station_location_profile_feat`
2. Reservation Insights (from reservation_txn_fact - numerical insights)
`reservation_txn_feat` or `user_reservation_feat`
Or more descriptive: `reservation_behavioral_feat`
3. App Usage Behavior (from app_log_fact - app using behavior)
`app_usage_behavior_feat` or `user_app_behavior_feat`
Or more concise: `app_behavior_feat`

## station_location_profile_feat

In [4]:
execution_date = '2025-11-16'
execution_date = get_execution_date(execution_date) if execution_date else datetime.now().strftime("%Y-%m-%d")
station_dim_df = load_data('data/curated/station_dim')
station_dim_df = station_dim_df.filter(F.col('dl_data_dt') == execution_date)

[ 2025-11-17 23:27:50 ] | churn_prediction | INFO     | loaders.py:load_data:170 | Loading data from local: data/curated/station_dim


In [8]:
from src.churn_prediction.constants.feature_engineering.station_location_profile_feat import (BUFFER_RADIUS,
                                                                                              poi_group_1_daily_life,
                                                                                              poi_group_2_shopping,
                                                                                              poi_group_3_leisure,
                                                                                              poi_group_4_travel_tourism)

In [9]:
poi_groups = {
    "daily_life": poi_group_1_daily_life,
    "shopping": poi_group_2_shopping,
    "leisure": poi_group_3_leisure,
    "travel_tourism": poi_group_4_travel_tourism
}

poi_tags = {tag: sub_tags for _, tags in poi_groups.items() for tag, sub_tags in tags.items()}

In [ ]:
# ========================================================================
# Geospatial Data Cleansing Module - Production-Ready Implementation
# Purpose: Enrich station data with province and district information
# Best Practices: Type hints, error handling, logging, reusability
# ========================================================================

from typing import Tuple, Dict, Any
from dataclasses import dataclass

import geopandas as gpd
import osmnx as ox
from shapely.geometry import Point
from tqdm import tqdm
import pandas as pd

@dataclass
class GeoConfig:
    """Configuration for geospatial operations."""
    crs_wgs84: str = "EPSG:4326"
    province_shapefile: str = "research/geoBoundaries/geoBoundaries-THA-ADM1_simplified.shp"
    district_shapefile: str = "research/geoBoundaries/geoBoundaries-THA-ADM2_simplified.shp"
    shapename_column: str = "shapeName"
    predicate: str = "within"   
    
class GeoDataCleaner:
    """
    Production-grade geospatial data cleansing for station locations.
    
    Handles enrichment of station coordinates with administrative boundaries
    (provinces and districts) using spatial joins.
    
    Attributes:
        config (GeoConfig): Configuration settings
        df (DataFrame): Station data to enrich
    """
    
    # Columns to remove after spatial join (metadata columns)
    SPATIAL_JOIN_CLEANUP_COLS = [
        "index_right", "shapeISO", "shapeID", "shapeGroup", "shapeType"
    ]
    
    def __init__(self, df: DataFrame, config: GeoConfig = None):
        """
        Initialize GeoDataCleaner.
        
        Args:
            df: DataFrame with station data containing longitude/latitude
            config: GeoConfig instance with paths and settings
        """
        self.df = df.toPandas()  # Convert to Pandas for GeoPandas compatibility
        self.config = config or GeoConfig()
    
    def create_geodataframe(self) -> gpd.GeoDataFrame:
        """
        Create GeoDataFrame from station coordinates.
        
        Returns:
            GeoDataFrame with Point geometries
        """
        gdf = gpd.GeoDataFrame(
            self.df,
            geometry=gpd.points_from_xy(
                self.df.longitude, 
                self.df.latitude
            ),
            crs=self.config.crs_wgs84
        )
        return gdf
    
    def load_shapefile(self, shapefile_path: str, name: str = "") -> gpd.GeoDataFrame:
        """
        Load and validate shapefile.
        
        Args:
            shapefile_path: Path to shapefile
            name: Name for logging
            
        Returns:
            GeoDataFrame with shapefile data
            
        Raises:
            FileNotFoundError: If shapefile not found
        """
        try:
            gdf = gpd.read_file(shapefile_path)
            gdf = gdf.to_crs(self.config.crs_wgs84)
            logger.info(f"✓ Loaded {name}: {len(gdf)} features")
            return gdf
        except FileNotFoundError:
            logger.error(f"✗ Shapefile not found: {shapefile_path}")
            raise
        except Exception as e:
            logger.error(f"✗ Error loading shapefile: {str(e)}")
            raise
    
    def perform_spatial_join(
        self, 
        gdf: gpd.GeoDataFrame, 
        boundaries: gpd.GeoDataFrame,
        boundary_name: str
    ) -> gpd.GeoDataFrame:
        """
        Perform spatial join between stations and boundaries.
        
        Args:
            gdf: GeoDataFrame with station points
            boundaries: GeoDataFrame with boundary polygons
            boundary_name: Name of boundary (for logging)
            
        Returns:
            GeoDataFrame with joined data
        """
        joined = gpd.sjoin(
            gdf,
            boundaries,
            how="left",
            predicate=self.config.predicate
        )
        
        logger.info(f"✓ Spatial join with {boundary_name}: {joined['station_id'].nunique()} enriched")
        return joined
    
    def clean_column_name(self, df: DataFrame, name_column: str, suffix: str) -> DataFrame:
        """
        Rename and clean geographic name column.
        
        Args:
            df: DataFrame to clean
            name_column: Column containing geographic names
            suffix: Suffix to remove from names (e.g., ' Province')
            
        Returns:
            DataFrame with cleaned column
        """
        df_copy = df.copy()
        
        # Determine output column name
        output_col = 'province' if suffix == ' Province' else 'district'
        
        # Rename if column exists
        if name_column in df_copy.columns:
            df_copy = df_copy.rename(columns={name_column: output_col})
        
        # Clean the column - ensure we're working with a Series
        if output_col in df_copy.columns:
            df_copy[output_col] = (
                df_copy[output_col]
                .astype(str)
                .str.replace(suffix, '', regex=False)
                .str.strip()
                .replace('nan', 'Unknown')
            )
        
        logger.info(f"✓ Cleaned {output_col}: {df_copy[output_col].nunique()} unique values")
        
        return df_copy
    
    def remove_unnecessary_columns(
        self, 
        df: DataFrame, 
        cols_to_remove: list = None
    ) -> DataFrame:
        """
        Remove unnecessary columns from dataframe.
        
        Args:
            df: DataFrame to clean
            cols_to_remove: List of columns to remove
            
        Returns:
            DataFrame with cleaned columns
        """
        cols_to_remove = cols_to_remove or self.SPATIAL_JOIN_CLEANUP_COLS
        existing_cols = [col for col in cols_to_remove if col in df.columns]
        
        if existing_cols:
            df_copy = df.drop(columns=existing_cols, errors='ignore')
            logger.info(f"✓ Removed {len(existing_cols)} metadata columns")
            return df_copy
        
        return df
    
    def enrich_with_province(self) -> 'GeoDataCleaner':
        """
        Enrich station data with province information.
        
        Returns:
            Self for method chaining
        """
        logger.info("\n" + "="*70)
        logger.info("STEP 1: Enriching with Province Data")
        logger.info("="*70)
        
        gdf = self.create_geodataframe()
        provinces = self.load_shapefile(
            self.config.province_shapefile, 
            "Province shapefile"
        )
        
        joined = self.perform_spatial_join(gdf, provinces, "provinces")
        self.df = self.clean_column_name(
            joined, 
            self.config.shapename_column, 
            ' Province'
        )
        self.df = self.remove_unnecessary_columns(
            self.df,
            self.SPATIAL_JOIN_CLEANUP_COLS + ['geometry']
        )
        
        return self  # Enable method chaining
    
    def enrich_with_district(self) -> 'GeoDataCleaner':
        """
        Enrich station data with district information.
        
        Returns:
            Self for method chaining
        """
        logger.info("\n" + "="*70)
        logger.info("STEP 2: Enriching with District Data")
        logger.info("="*70)
        
        gdf = self.create_geodataframe()
        districts = self.load_shapefile(
            self.config.district_shapefile,
            "District shapefile"
        )
        
        joined = self.perform_spatial_join(gdf, districts, "districts")
        self.df = self.clean_column_name(
            joined,
            self.config.shapename_column,
            ' District'
        )

        self.df.rename(columns={'geometry': 'locationgeometry'}, inplace=True)
        
        # Join geometry for potential future spatial analysis
        if 'shapeID' in self.df.columns:
            self.df = self.df.join(
                districts[['geometry', 'shapeID']].set_index('shapeID'),
                on='shapeID',
                how='left'
            )

        self.df = self.remove_unnecessary_columns(
            self.df,
            [col for col in self.SPATIAL_JOIN_CLEANUP_COLS if col != 'shapeID']
        )
        
        return self
    
    def get_result(self) -> DataFrame:
        """
        Get cleaned dataframe.
        
        Returns:
            Cleaned station DataFrame
        """
        logger.info("\n" + "="*70)
        logger.info("RESULT SUMMARY")
        logger.info("="*70)
        logger.info(f"Total stations: {len(self.df)}")
        
        return self.df


# ========================================================================
# Execute Geospatial Cleansing Pipeline
# ========================================================================

logger.info("\nGEOSPATIAL DATA CLEANSING PIPELINE - Starting...\n")

# Initialize cleaner and run pipeline with method chaining
df = (
    GeoDataCleaner(station_dim_df)
    .enrich_with_province()
    .enrich_with_district()
    .get_result()
)
geometry_list = df.geometry.drop_duplicates().reset_index(drop=True)

logger.info("\n✓ Geospatial cleansing completed successfully!\n")

INFO - 
GEOSPATIAL DATA CLEANSING PIPELINE - Starting...

INFO - 
INFO - STEP 1: Enriching with Province Data
INFO - ======================================================================
INFO - ✓ Loaded Province shapefile: 77 features
INFO - ✓ Spatial join with provinces: 1243 enriched
INFO - ✓ Cleaned province: 39 unique values
INFO - ✓ Removed 6 metadata columns
INFO - 
INFO - STEP 2: Enriching with District Data
INFO - ======================================================================
INFO - ✓ Loaded District shapefile: 928 features
INFO - ✓ Spatial join with districts: 1243 enriched
INFO - ✓ Cleaned district: 122 unique values
INFO - ✓ Removed 4 metadata columns
INFO - 
INFO - RESULT SUMMARY
INFO - ======================================================================
INFO - Total stations: 1243
INFO - 
✓ Geospatial cleansing completed successfully!



In [42]:
# ========================================================================
# POI Analysis Module - Production-Ready Implementation
# Best Practices: Type hints, logging, caching, comprehensive docs
# ========================================================================

from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass
import warnings

@dataclass
class POIConfig:
    """Configuration for POI analysis."""
    buffer_radius: int = BUFFER_RADIUS
    crs_wgs84: str = "EPSG:4326"
    crs_utm: str = "EPSG:32647"
    predicate: str = "within"


class POIAnalyzer:
    """
    A production-grade POI (Point of Interest) analyzer for spatial analysis.
    
    This class provides methods to count and analyze POIs around stations,
    with support for grouping, percentage calculations, and error handling.
    
    Attributes:
        config (POIConfig): Configuration settings for analysis
    """
    
    def __init__(self, config: Optional[POIConfig] = None):
        """Initialize POI Analyzer with optional configuration."""
        self.config = config or POIConfig()
        self.validate_config()
    
    def validate_config(self) -> None:
        """Validate configuration parameters."""
        if self.config.buffer_radius <= 0:
            raise ValueError("buffer_radius must be positive")
    
    def count_pois_by_groups(
        self, 
        joined_gdf: 'gpd.GeoDataFrame', 
        poi_groups_dict: Dict[str, Dict[str, List[str]]]
    ) -> pd.DataFrame:
        """
        Count POIs by groups around each station.
        
        Args:
            joined_gdf: Spatial join result with POI and station data
            poi_groups_dict: POI group definitions {group_name: {tag: [values]}}
            
        Returns:
            DataFrame with columns [station_id, poi_cnt_{group_name}]
            
        Raises:
            ValueError: If inputs are empty or invalid
            KeyError: If required columns are missing
            
        Example:
            >>> analyzer = POIAnalyzer()
            >>> result = analyzer.count_pois_by_groups(joined_gdf, poi_groups)
            >>> print(result.head())
        """
        self.validate_inputs(joined_gdf, poi_groups_dict)
        
        results = []
        
        try:
            for station_id in joined_gdf['station_id'].unique():
                station_data = joined_gdf[joined_gdf['station_id'] == station_id]
                row = {'station_id': station_id}
                
                for group_name, group_tags in poi_groups_dict.items():
                    group_count = self.count_group_pois(station_data, group_tags)
                    row[f'poi_cnt_{group_name}'] = group_count
                
                results.append(row)
        
        except Exception as e:
            raise RuntimeError(f"Error processing POI groups: {str(e)}")
        
        return pd.DataFrame(results)
    
    def validate_inputs(
        self, 
        joined_gdf: 'gpd.GeoDataFrame', 
        poi_groups_dict: Dict
    ) -> None:
        """Validate input data and structure."""
        if joined_gdf.empty:
            raise ValueError("joined_gdf cannot be empty")
        if not poi_groups_dict:
            raise ValueError("poi_groups_dict cannot be empty")
        if 'station_id' not in joined_gdf.columns:
            raise KeyError("'station_id' column not found in joined_gdf")
    
    @staticmethod
    def count_group_pois(
        station_data: pd.DataFrame, 
        group_tags: Dict[str, List[str]]
    ) -> int:
        """
        Count POIs in a specific group for a station.
        
        Args:
            station_data: Station-specific POI data
            group_tags: Tags defining the POI group {tag_key: [values]}
            
        Returns:
            Total count of POIs matching the group criteria
        """
        count = 0
        for tag_key, tag_values in group_tags.items():
            if tag_key not in station_data.columns:
                continue
            count += station_data[tag_key].isin(tag_values).sum()
        return count
    
    def calculate_poi_percentages(
        self, 
        poi_summary_df: pd.DataFrame, 
        poi_cols: List[str],
        suffix: str = 'perc_'
    ) -> pd.DataFrame:
        """
        Calculate percentage distribution of POI groups.
        
        Args:
            poi_summary_df: DataFrame with POI counts
            poi_cols: Column names to calculate percentages for
            suffix: Prefix for percentage column names
            
        Returns:
            DataFrame with original and percentage columns
            
        Note:
            Handles zero-division gracefully with fillna(0)
        """
        result_df = poi_summary_df.copy()
        result_df['sum_poi'] = result_df[poi_cols].sum(axis=1)
        
        for col in poi_cols:
            # Avoid division by zero
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                result_df[f'{suffix}{col}'] = (
                    result_df[col] / result_df['sum_poi']
                ).fillna(0).round(4)
        
        return result_df
    
    def extract_poi_features(
        self, 
        station_gps_df: 'gpd.GeoDataFrame', 
        pois: 'gpd.GeoDataFrame'
    ) -> 'gpd.GeoDataFrame':
        """
        Extract POI features within buffer radius around stations.
        
        Args:
            station_gps_df: Stations with geometries (WGS84)
            pois: POI points (WGS84)
            
        Returns:
            Spatial join result of POIs within buffer
            
        Note:
            Automatically converts to UTM for accurate distance calculations
        """
        try:
            # Convert to projected CRS for accurate distances
            station_buffer = station_gps_df.to_crs(self.config.crs_utm).copy()
            station_buffer.geometry = station_buffer.geometry.buffer(
                self.config.buffer_radius
            )
            
            pois_utm = pois.to_crs(self.config.crs_utm)
            
            # Spatial join
            joined = gpd.sjoin(
                pois_utm, 
                station_buffer, 
                how="inner", 
                predicate=self.config.predicate
            )
            
            return joined
        
        except Exception as e:
            raise RuntimeError(f"Error extracting POI features: {str(e)}")
    
    def identify_dominant_poi_type(
        self,
        summary_df: pd.DataFrame
    ) -> pd.DataFrame:
        """
        Identify the dominant POI type for each station based on max percentage.
        
        Args:
            summary_df: DataFrame with POI percentages (from calculate_poi_percentages)
            poi_groups_dict: POI group definitions for mapping group names
            
        Returns:
            DataFrame with added 'dominant_poi_type' column
            
        Example:
            >>> summary_with_type = analyzer.identify_dominant_poi_type(summary_df, poi_groups)
            >>> print(summary_with_type[['station_id', 'dominant_poi_type']])
        """
        df_copy = summary_df.copy()
        
        # Get percentage columns
        perc_cols = [col for col in df_copy.columns if col.startswith('perc_poi_cnt_')]
        
        if not perc_cols:
            raise ValueError("No percentage columns found. Run calculate_poi_percentages first.")
        
        # Extract group names from column names (e.g., 'perc_poi_cnt_daily_life' -> 'daily_life')
        group_names = [col.replace('perc_poi_cnt_', '') for col in perc_cols]
        
        # Find dominant type for each station
        def get_dominant_type(row):
            max_perc = -1
            dominant = 'None'
            for group_name, perc_col in zip(group_names, perc_cols):
                if row[perc_col] > max_perc:
                    max_perc = row[perc_col]
                    dominant = group_name
            return dominant
        
        df_copy['dominant_poi_type'] = df_copy.apply(get_dominant_type, axis=1)
        
        return df_copy
    
    def run_full_pipeline(
        self,
        station_gps_df: 'gpd.GeoDataFrame',
        pois: 'gpd.GeoDataFrame',
        poi_groups_dict: Dict[str, Dict[str, List[str]]]
    ) -> Tuple[pd.DataFrame, 'gpd.GeoDataFrame']:
        """
        Run complete POI analysis pipeline.
        
        Args:
            station_gps_df: Stations with geometries
            pois: POI points
            poi_groups_dict: POI group definitions
            
        Returns:
            Tuple of (summary_df, joined_gdf) where summary_df includes dominant_poi_type
        """
        # Extract features
        joined = self.extract_poi_features(station_gps_df, pois)
        if joined.empty:
            return pd.DataFrame(), gpd.GeoDataFrame()
        
        # Count POIs by groups
        summary_df = self.count_pois_by_groups(joined, poi_groups_dict)
        
        # Calculate percentages
        poi_cols = [col for col in summary_df.columns if col.startswith('poi_cnt_')]
        summary_df = self.calculate_poi_percentages(summary_df, poi_cols)
        
        # Identify dominant POI type
        summary_df = self.identify_dominant_poi_type(summary_df)
        
        return summary_df, joined

In [ ]:
# ========================================================================
# Execute POI Analysis Pipeline
# ========================================================================

# Initialize analyzer with configuration
config = POIConfig(buffer_radius=BUFFER_RADIUS)
analyzer = POIAnalyzer(config=config)

# Prepare data
poi_group_summary_list = []
expected_columns = ["name", "geometry"] + list(poi_tags.keys())

for geometry in tqdm(geometry_list):
    station_area_df = df[df["geometry"] == geometry]
    poi = ox.features_from_polygon(geometry, tags=poi_tags)
    pois = poi.reindex(columns=expected_columns).reset_index(drop=True)

    station_gps_df = gpd.GeoDataFrame(
        station_area_df,
        geometry=gpd.points_from_xy(
            station_area_df.longitude, 
            station_area_df.latitude
        ),
        crs="EPSG:4326"
    )

    # Run full pipeline
    poi_group_summary_df, joined = analyzer.run_full_pipeline(
        station_gps_df, 
        pois, 
        poi_groups
    )
    poi_group_summary_list.append(poi_group_summary_df)

poi_group_final_summary_df = pd.concat(poi_group_summary_list)

logger.info("✓ POI analysis completed successfully")

100%|██████████| 122/122 [03:34<00:00,  1.76s/it]

✓ POI analysis completed successfully


In [46]:
poi_group_final_summary_df

,station_id,poi_cnt_daily_life,poi_cnt_shopping,poi_cnt_leisure,poi_cnt_travel_tourism,sum_poi,perc_poi_cnt_daily_life,perc_poi_cnt_shopping,perc_poi_cnt_leisure,perc_poi_cnt_travel_tourism,dominant_poi_type
0,1,78,77,18,56,229,0.3406,0.3362,0.0786,0.2445,daily_life
1,2,59,60,33,29,181,0.3260,0.3315,0.1823,0.1602,shopping
2,50,78,81,27,54,240,0.3250,0.3375,0.1125,0.2250,shopping
3,63,86,62,20,54,222,0.3874,0.2793,0.0901,0.2432,daily_life
4,210,62,77,9,48,196,0.3163,0.3929,0.0459,0.2449,shopping
...,...,...,...,...,...,...,...,...,...,...,...
0,2663,1,0,0,0,1,1.0000,0.0000,0.0000,0.0000,daily_life
0,2679,8,5,0,5,18,0.4444,0.2778,0.0000,0.2778,daily_life
1,2687,8,5,0,5,18,0.4444,0.2778,0.0000,0.2778,daily_life
0,2684,5,6,0,1,12,0.4167,0.5000,0.0000,0.0833,shopping


## reservation_behavioral_feat

In [26]:
# load curated data
reservation_txn_fact_df = load_data('data/curated/reservation_txn_fact')
customer_profile_dim_df = load_data('data/curated/customer_profile_dim')
customer_group_dim_df = load_data('data/curated/customer_group_dim')
station_dim_df = load_data('data/curated/station_dim')
holiday_master_df = load_data('data/curated/holiday_master')

[ 2025-11-17 18:10:29 ] | churn_prediction | INFO     | loaders.py:load_data:170 | Loading data from local: data/curated/reservation_txn_fact
[ 2025-11-17 18:10:31 ] | churn_prediction | INFO     | loaders.py:load_data:170 | Loading data from local: data/curated/customer_profile_dim
[ 2025-11-17 18:10:31 ] | churn_prediction | INFO     | loaders.py:load_data:170 | Loading data from local: data/curated/customer_group_dim
[ 2025-11-17 18:10:31 ] | churn_prediction | INFO     | loaders.py:load_data:170 | Loading data from local: data/curated/station_dim
[ 2025-11-17 18:10:31 ] | churn_prediction | INFO     | loaders.py:load_data:170 | Loading data from local: data/curated/holiday_master


In [12]:
reservation_txn_fact_df.filter(F.col('group_id') == '0').count()

4366

In [ ]:
base_df = reservation_txn_fact_df.join(
    customer_profile_dim_df, on='user_id', how='left').join(
    customer_group_dim_df, on='group_id', how='left').join(
    station_dim_df, on='station_id', how='left'
    )
base_df = base_df.filter((F.col('user_id') != '0')
                        & (F.col('station_id') != '0')
                        & (~F.col('group_id').isin(['','1','2','4','330','356']))
                        & (F.col('admin_id') == '0')
                        )
base_df.show()

25/11/17 18:11:28 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----------+--------+-------+------+-----------------+--------------+------------+--------------+-------------------+-------------------+--------+----------+--------------+--------+-----------+------------+--------+-------------------+----------+--------------------+----------+----------+---------+--------+-----------+---------+----------+-------------------+----------+--------------------+----------+----------+----------+----------+--------------------+--------------+--------------------+----------+----------+----------+--------------------+
|station_id|group_id|user_id|txn_id|reservation_state|payment_method|promotion_id|promotion_code| reserve_start_time|  reserve_stop_time|distance|hour_price|distance_price|discount|total_price|total_charge|category|             txn_ts|dl_data_dt|          dl_load_ts|first_name| last_name|activated|admin_id|        sex|foreigner| birthdate|      registed_time|dl_data_dt|          dl_load_ts|group_code|group_name|dl_data_dt|dl_load_ts|        statio

## app_behavior_feat